# Million Dice Roll Statistics Simulator - Extended Project (Solution)

**Based on Al Sweigart's *The Big Book of Small Python Projects* – Project #46**

This notebook implements and **extends** the original million-dice-roll simulator with:

- Generalization to any number of dice and any number of sides
- Progress reporting during the long simulation
- Clean tabular output of counts and percentages
- Matplotlib bar-chart visualization of the empirical distribution
- Two alternate implementations (collections.Counter and NumPy vectorized)
- Exact theoretical probabilities via dynamic programming
- Side-by-side empirical vs theoretical comparison
- Configurable simulation experiments (More Practice + Simulation section)

Use the matching **Practice Skeleton** notebook to implement the pieces yourself first.


In [ ]:
from IPython.display import Image, display
display(Image(filename='million_dice_flowchart.png', width=950))
print("Flowchart of the Million Dice Roll Statistics Simulator (core path + alternates).")


## 0. Imports & Reproducibility

We import everything we need once at the top. A fixed random seed makes the demo runs reproducible while still illustrating randomness.


In [ ]:
import random
import time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# Make the educational runs reproducible (comment out for pure randomness)
random.seed(42)
np.random.seed(42)

print("Libraries loaded.")


## 1. Core Simulation Function (matches original book logic)

`simulate_dice_rolls(number_of_dice, sides=6, num_simulations=1_000_000, show_progress=True)`

- Initializes a results dictionary for every possible sum from `number_of_dice` to `number_of_dice * sides`.
- Performs the Monte-Carlo loop, updating counts.
- Optionally prints progress every second (exactly like the original program).
- Returns the results dictionary.


In [ ]:
def simulate_dice_rolls(number_of_dice, sides=6, num_simulations=1_000_000, show_progress=True):
    """Simulate rolling `number_of_dice` dice with `sides` faces, `num_simulations` times.
    Returns a dict: total -> count.
    """
    # Possible totals range from all 1s to all `sides`
    min_total = number_of_dice * 1
    max_total = number_of_dice * sides
    results = {total: 0 for total in range(min_total, max_total + 1)}

    if show_progress:
        print(f'Simulating {num_simulations:,} rolls of {number_of_dice} {sides}-sided dice...')
    last_print_time = time.time()

    for i in range(num_simulations):
        if show_progress and time.time() > last_print_time + 1:
            pct = round(i / (num_simulations / 100), 1)
            print(f'{pct}% done...')
            last_print_time = time.time()

        total = 0
        for _ in range(number_of_dice):
            total += random.randint(1, sides)
        results[total] += 1

    return results


# Demo with the classic 2d6 case (small number of sims for speed in notebook)
demo_results = simulate_dice_rolls(2, sides=6, num_simulations=100_000, show_progress=True)
print("Sample counts for 2d6 (100k rolls):", dict(list(demo_results.items())[:5]), "...")


## 2. Display Results Table

Pretty-print the classic “TOTAL – ROLLS – PERCENTAGE” table.  
Percentage = count / num_simulations * 100, rounded to 1 decimal (identical to the book’s `/ 10000` trick when num_simulations = 1 000 000).


In [ ]:
def display_results(results, num_simulations=1_000_000):
    """Print the classic TOTAL - ROLLS - PERCENTAGE table."""
    print('TOTAL - ROLLS - PERCENTAGE')
    for total in sorted(results.keys()):
        count = results[total]
        percentage = round(count / num_simulations * 100, 1)
        print(f' {total} - {count} rolls - {percentage}%')


# Show the 100k demo table
print("=== 2 six-sided dice, 100 000 simulations ===")
display_results(demo_results, num_simulations=100_000)


## 3. Visualization – Empirical Distribution Bar Chart

A bar chart makes the characteristic shape of dice-sum distributions obvious (triangular for 2d6, approaching normal for larger N by the Central Limit Theorem).


In [ ]:
def plot_distribution(results, number_of_dice, sides, num_simulations, title_suffix=""):
    """Bar chart of percentage for each possible total."""
    totals = sorted(results.keys())
    percentages = [results[t] / num_simulations * 100 for t in totals]

    plt.figure(figsize=(10, 5))
    bars = plt.bar(totals, percentages, color='steelblue', edgecolor='navy', alpha=0.85)
    plt.xlabel('Sum of dice', fontsize=12)
    plt.ylabel('Percentage (%)', fontsize=12)
    plt.title(f'Empirical distribution of {number_of_dice}d{sides} ({num_simulations:,} rolls){title_suffix}', fontsize=14)
    plt.xticks(totals)
    plt.grid(axis='y', alpha=0.3)
    # annotate the mode
    mode = max(results, key=results.get)
    plt.annotate(f'mode={mode}', xy=(mode, results[mode]/num_simulations*100),
                 xytext=(mode+0.5, results[mode]/num_simulations*100 + 1),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red')
    plt.tight_layout()
    plt.savefig('million_dice_distribution.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Chart saved as million_dice_distribution.png")


plot_distribution(demo_results, 2, 6, 100_000)


## 4. Alternate Implementation #1 – collections.Counter

Instead of a pre-initialized dict we can let `Counter` discover the keys. Functionally identical, slightly more compact.


In [ ]:
def simulate_with_counter(number_of_dice, sides=6, num_simulations=1_000_000):
    """Same simulation using collections.Counter."""
    counter = Counter()
    for _ in range(num_simulations):
        total = sum(random.randint(1, sides) for _ in range(number_of_dice))
        counter[total] += 1
    # Ensure every possible key exists (even if 0) for a clean table
    min_t, max_t = number_of_dice, number_of_dice * sides
    for t in range(min_t, max_t + 1):
        if t not in counter:
            counter[t] = 0
    return dict(counter)


# Quick verification that Counter version matches original style
c_results = simulate_with_counter(2, 6, 50_000)
print("Counter version (50k rolls) first few:", dict(list(c_results.items())[:4]))
print("Total rolls accounted for:", sum(c_results.values()))


## 5. Alternate Implementation #2 – NumPy Vectorized (fast)

For educational speed-ups we generate the entire (num_simulations × number_of_dice) matrix at once and sum along the dice axis. Orders of magnitude faster for large simulations.


In [ ]:
def simulate_numpy(number_of_dice, sides=6, num_simulations=1_000_000):
    """Vectorized simulation with NumPy. Returns dict total -> count."""
    # shape = (num_simulations, number_of_dice)
    rolls = np.random.randint(1, sides + 1, size=(num_simulations, number_of_dice))
    totals = rolls.sum(axis=1)
    unique, counts = np.unique(totals, return_counts=True)
    results = {int(u): int(c) for u, c in zip(unique, counts)}
    # fill missing keys with 0
    for t in range(number_of_dice, number_of_dice * sides + 1):
        results.setdefault(t, 0)
    return results


# Speed comparison (small demo)
import time as _t
t0 = _t.time()
_ = simulate_dice_rolls(3, 6, 200_000, show_progress=False)
t_loop = _t.time() - t0
t0 = _t.time()
_ = simulate_numpy(3, 6, 200_000)
t_np = _t.time() - t0
print(f"Pure-Python loop 200k rolls of 3d6: {t_loop:.3f}s")
print(f"NumPy vectorized 200k rolls of 3d6: {t_np:.3f}s")
print(f"Speed-up ≈ {t_loop / t_np:.1f}×")


## 6. Exact Theoretical Probabilities (Dynamic Programming)

For any reasonable N and S we can compute the *exact* probability of every sum with a classic DP recurrence:

`dp[d][s] = sum(dp[d-1][s-f] for f in 1..sides)`  

with base case `dp[0][0] = 1`.

This lets us compare the Monte-Carlo percentages against the true mathematical values.


In [ ]:
def theoretical_probs(number_of_dice, sides=6):
    """Return dict: total -> exact probability (float)."""
    # dp[d][s] = number of ways to obtain sum s with d dice
    max_sum = number_of_dice * sides
    dp = [0] * (max_sum + 1)
    dp[0] = 1  # 0 dice → sum 0 in 1 way

    for d in range(1, number_of_dice + 1):
        new_dp = [0] * (max_sum + 1)
        for prev_sum, ways in enumerate(dp):
            if ways == 0:
                continue
            for face in range(1, sides + 1):
                new_dp[prev_sum + face] += ways
        dp = new_dp

    total_ways = sides ** number_of_dice
    return {s: dp[s] / total_ways for s in range(number_of_dice, max_sum + 1) if dp[s] > 0}


# Exact 2d6 probabilities (classic)
theo_2d6 = theoretical_probs(2, 6)
print("Exact 2d6 probabilities:")
for t, p in sorted(theo_2d6.items()):
    print(f"  {t}: {p*100:.1f}%  ({int(p * 36)}/36)")


## 7. Empirical vs Theoretical Comparison

Run a larger simulation and overlay the exact curve on the bar chart.


In [ ]:
def compare_empirical_theoretical(number_of_dice=2, sides=6, num_simulations=500_000):
    emp = simulate_numpy(number_of_dice, sides, num_simulations)
    theo = theoretical_probs(number_of_dice, sides)

    totals = sorted(emp.keys())
    emp_pct = [emp[t] / num_simulations * 100 for t in totals]
    theo_pct = [theo.get(t, 0) * 100 for t in totals]

    plt.figure(figsize=(10, 5))
    width = 0.4
    x = np.array(totals)
    plt.bar(x - width/2, emp_pct, width, label='Empirical (Monte-Carlo)', color='steelblue', alpha=0.8)
    plt.bar(x + width/2, theo_pct, width, label='Theoretical (exact)', color='orange', alpha=0.8)
    plt.xlabel('Sum')
    plt.ylabel('Percentage (%)')
    plt.title(f'{number_of_dice}d{sides}: Empirical ({num_simulations:,} rolls) vs Exact')
    plt.legend()
    plt.xticks(totals)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('million_dice_emp_vs_theo.png', dpi=120, bbox_inches='tight')
    plt.show()

    # max absolute error
    max_err = max(abs(e - t) for e, t in zip(emp_pct, theo_pct))
    print(f"Max absolute percentage error: {max_err:.3f} percentage points")


compare_empirical_theoretical(2, 6, 500_000)


## 8. More Practice Exercises

Try the following yourself (solutions already shown for reference):

1. Simulate 1 000 000 rolls of **four 6-sided dice** and locate the mode.
2. Change to **8-sided**, **10-sided** or **20-sided** dice.
3. Model a fair coin as a 2-sided “die” and simulate 100 000 tosses of 10 coins (number of heads).
4. Use the theoretical function to confirm that two fair coins give P(exactly 1 head) = 50 %.


In [ ]:
# Practice 1 – 4d6
print("=== Practice 1: 4d6 (200 000 rolls for speed) ===")
res_4d6 = simulate_numpy(4, 6, 200_000)
display_results(res_4d6, 200_000)
mode_4d6 = max(res_4d6, key=res_4d6.get)
print(f"Mode of 4d6 ≈ {mode_4d6}")

# Practice 2 – d20
print("\n=== Practice 2: single d20 (100 000 rolls) ===")
res_d20 = simulate_numpy(1, 20, 100_000)
# should be almost flat ~5 %
print("Percentages for faces 1,10,20:",
      round(res_d20[1]/1000,1), round(res_d20[10]/1000,1), round(res_d20[20]/1000,1))

# Practice 3 – 10 coins (2-sided)
print("\n=== Practice 3: 10 fair coins (number of heads) ===")
res_coins = simulate_numpy(10, 2, 100_000)
display_results(res_coins, 100_000)

# Practice 4 – theoretical two coins
print("\n=== Practice 4: exact P(exactly 1 head) with 2 coins ===")
theo_coins = theoretical_probs(2, 2)
print(f"P(sum=2) i.e. exactly 1 head? Wait – faces are 1 or 2, so sum=3 means one of each:")
print({k: f'{v*100:.1f}%' for k,v in theo_coins.items()})
print("(sum=3 corresponds to one 1 and one 2 → 50 % as expected)"))


## 9. Simulation Section – Parameter Sweeps

Change any of the parameters below and re-run the cell to observe how the shape, mean and variance of the sum distribution evolve.  
By the Central Limit Theorem, larger numbers of dice produce an approximately normal curve.


In [ ]:
def run_simulation_experiment(configs):
    """configs = list of (n_dice, sides, n_sims, label)"""
    fig, axes = plt.subplots(1, len(configs), figsize=(5*len(configs), 4), sharey=True)
    if len(configs) == 1:
        axes = [axes]

    for ax, (n, s, sims, label) in zip(axes, configs):
        res = simulate_numpy(n, s, sims)
        totals = sorted(res.keys())
        pct = [res[t]/sims*100 for t in totals]
        ax.bar(totals, pct, color='teal', alpha=0.8, edgecolor='darkgreen')
        mean = sum(t * res[t] for t in totals) / sims
        ax.axvline(mean, color='red', linestyle='--', label=f'mean≈{mean:.1f}')
        ax.set_title(label)
        ax.set_xlabel('Sum')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

    axes[0].set_ylabel('Percentage (%)')
    plt.suptitle('Dice-sum distributions under different parameters', fontsize=14)
    plt.tight_layout()
    plt.savefig('million_dice_simulation_sweep.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Sweep chart saved as million_dice_simulation_sweep.png")


# Example sweep – feel free to edit the list
experiments = [
    (1, 6, 50_000, '1d6 (flat)'),
    (2, 6, 100_000, '2d6 (triangle)'),
    (5, 6, 200_000, '5d6 (≈normal)'),
    (10, 6, 200_000, '10d6 (very normal)'),
]
run_simulation_experiment(experiments)


## 10. Full Classic Run (optional – takes ~10-20 s)

Uncomment the cell below to reproduce the exact original-book interaction with 1 000 000 rolls of 2d6.  
(Progress messages appear once per second.)


In [ ]:
# Uncomment to run the classic million-roll simulation:
# classic = simulate_dice_rolls(2, sides=6, num_simulations=1_000_000, show_progress=True)
# display_results(classic, 1_000_000)
# plot_distribution(classic, 2, 6, 1_000_000, title_suffix=" – classic book example")

print("Classic full-million cell is commented out for notebook responsiveness.")
print("Uncomment the lines above when you want the authentic slow progress output.")


## Key Takeaways (Solution)

- Monte-Carlo simulation is a practical way to obtain probabilities when the analytic formula is cumbersome.
- Pre-allocating the results dictionary (or using Counter / NumPy) keeps the code clear and efficient.
- The distribution of the sum of independent dice rapidly approaches a normal curve (Central Limit Theorem).
- Exact probabilities can still be computed cheaply with a simple dynamic-programming recurrence for moderate N and S.
- Multiple equivalent implementations (loop, Counter, vectorized) help learners understand the same idea from different angles.
- Parameter sweeps make the effect of “more dice → more normal” immediately visible.
